# 01 — Chargement et contrôle du jeu BIRD Mini-Dev

Ce notebook décrit les **500 questions déjà préparées** utilisées dans l'étude. Les bases SQLite sont conservées dans `data/raw/bird/dev_databases` et ne sont pas relues ici.

Objectifs : vérifier la qualité du jeu de questions, les bases couvertes et la disponibilité de l'evidence métier.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
QUESTIONS_PATH = ROOT / 'data' / 'processed' / 'questions.json'
with QUESTIONS_PATH.open(encoding='utf-8') as stream:
    questions = json.load(stream)
df = pd.DataFrame(questions)
print(f'{len(df)} questions chargées depuis {QUESTIONS_PATH.relative_to(ROOT)}')
df.head(3)

In [ ]:
required = ['question_id', 'db_id', 'question', 'sql_gold', 'evidence']
assert set(required).issubset(df.columns), f'Colonnes absentes : {set(required) - set(df.columns)}'
quality = pd.Series({
    'questions': len(df),
    'question_id uniques': df.question_id.nunique(),
    'bases distinctes': df.db_id.nunique(),
    'SQL gold vide': int(df.sql_gold.fillna('').str.strip().eq('').sum()),
    'evidence vide': int(df.evidence.fillna('').str.strip().eq('').sum()),
})
quality.to_frame('valeur')

In [ ]:
by_database = df.groupby('db_id').size().sort_values(ascending=False)
display(by_database.to_frame('nombre de questions'))
ax = by_database.plot.bar(figsize=(11, 4), color='#3b82f6')
ax.set(title='Répartition des questions par base BIRD', xlabel='Base', ylabel='Questions')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()

In [ ]:
DB_ROOT = ROOT / 'data' / 'raw' / 'bird' / 'dev_databases'
available = {path.parent.name for path in DB_ROOT.glob('*/*.sqlite')}
expected = set(df.db_id)
pd.DataFrame({
    'bases attendues': sorted(expected),
    'sqlite trouvée': [db_id in available for db_id in sorted(expected)],
})

## Conclusion

Le jeu d'étude est valide si les 500 identifiants sont uniques, si chaque question possède un SQL de référence et si toutes les bases attendues sont présentes. Les sorties servent directement aux notebooks 02 à 09.